In [2]:
"""
Programmatic Flip Rate Reduction Test
Simulate original inconsistency and test if QDs reduce flip rate
"""

import time
import json
import csv
import random
from collections import defaultdict
from datetime import datetime
import mlflow
import mlflow.sklearn

# API Configuration
BASE_URL = "http://ece-nebula16.eng.uwaterloo.ca:11434"

# MLflow configuration
mlflow.set_tracking_uri("file:./mlruns")  # Use local file storage

In [1]:
%pip install -q mlflow

Note: you may need to restart the kernel to use updated packages.


In [3]:
def generate_with_delay(prompt: str, reasoning: bool = False, delay: float = 1.0) -> str:
    """Call LLM API with exponential backoff delay"""
    import requests
    
    payload = {
        "model": "gpt-oss:120b",
        "prompt": prompt,
        "stream": False
    }
    
    if delay > 0:
        time.sleep(delay)
    
    try:
        response = requests.post(
            f"{BASE_URL}/api/generate", 
            headers={"Content-Type": "application/json"},
            json=payload,
            timeout=60
        )
        
        if response.status_code == 200:
            result = response.json()
            return result.get("response", "No response returned")
        else:
            print(f"API Error: {response.status_code} - {response.text}")
            return "API Error occurred"
    except Exception as e:
        print(f"Connection Error: {e}")
        return "Connection Error occurred"

In [ ]:
import pandas as pd

print("Loading problematic data:")
initial_df = pd.read_csv('./bad_questions.csv', delimiter='\t')

initial_df.head(5)

initial_df.columns


Loading problematic data:


Index(['username', 'lab_number', 'question_number', 'ts', 'question_text',
       'grade', 'term'],
      dtype='object')

In [5]:
# Group by lab number and username 
lab_1_df = initial_df[initial_df['lab_number'] == 1].copy()
lab_2_df = initial_df[initial_df['lab_number'] == 2].copy()
lab_3_df = initial_df[initial_df['lab_number'] == 3].copy()
lab_4_df = initial_df[initial_df['lab_number'] == 4].copy()
lab_5_df = initial_df[initial_df['lab_number'] == 5].copy()


print(lab_1_df.info())



<class 'pandas.core.frame.DataFrame'>
Index: 430 entries, 0 to 1735
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   username         430 non-null    object
 1   lab_number       430 non-null    int64 
 2   question_number  430 non-null    int64 
 3   ts               430 non-null    object
 4   question_text    430 non-null    object
 5   grade            430 non-null    int64 
 6   term             430 non-null    object
dtypes: int64(3), object(4)
memory usage: 26.9+ KB
None


In [6]:
def calculate_flip_rate(df):
    """
    Calculate flip rate: percentage of unique question_text entries with grade disagreements
    across submissions.
    """
    flip_count = 0
    total_questions = 0
    
    grouped = df.groupby('question_text')
    
    for question_text, group in grouped:
        if len(group) > 1:  # Only consider questions with multiple submissions
            total_questions += 1
            if group['grade'].nunique() > 1:  # Check for grade disagreements
                flip_count += 1
    
    flip_rate = (flip_count / total_questions * 100) if total_questions > 0 else 0
    
    return {
        'flip_count': flip_count,
        'total_questions': total_questions,
        'flip_rate': flip_rate
    }

# Calculate flip rates
initial_stats = calculate_flip_rate(initial_df)
lab_1_stats = calculate_flip_rate(lab_1_df)
lab_2_stats = calculate_flip_rate(lab_2_df)
lab_3_stats = calculate_flip_rate(lab_3_df)
lab_4_stats = calculate_flip_rate(lab_4_df)
lab_5_stats = calculate_flip_rate(lab_5_df)
print(f"All Labs: {initial_stats['flip_rate']:.2f}% ({initial_stats['flip_count']}/{initial_stats['total_questions']})")
print(f"Lab 1: {lab_1_stats['flip_rate']:.2f}% ({lab_1_stats['flip_count']}/{lab_1_stats['total_questions']})")
print(f"Lab 2: {lab_2_stats['flip_rate']:.2f}% ({lab_2_stats['flip_count']}/{lab_2_stats['total_questions']})")
print(f"Lab 3: {lab_3_stats['flip_rate']:.2f}% ({lab_3_stats['flip_count']}/{lab_3_stats['total_questions']})")
print(f"Lab 4: {lab_4_stats['flip_rate']:.2f}% ({lab_4_stats['flip_count']}/{lab_4_stats['total_questions']})")
print(f"Lab 5: {lab_5_stats['flip_rate']:.2f}% ({lab_5_stats['flip_count']}/{lab_5_stats['total_questions']})")



All Labs: 100.00% (379/379)
Lab 1: 100.00% (93/93)
Lab 2: 100.00% (117/117)
Lab 3: 100.00% (54/54)
Lab 4: 100.00% (72/72)
Lab 5: 100.00% (43/43)


In [7]:
# Create dictionaries mapping (lab_number, question_number) to rubrics and context
rubrics_dict = {}
rubric_context_dict = {}  # Stores context (lab description, question) for each rubric

# Read each rubric file and parse the rubrics
rubric_files = {
    1: './Rubrics/rubric_lab1 1.txt',
    2: './Rubrics/rubric_lab2 1.txt',
    3: './Rubrics/rubric_lab3 1.txt',
    4: './Rubrics/rubric_lab4 1.txt',
    5: './Rubrics/rubric_lab5 1.txt'
}

import re

for lab_num, filename in rubric_files.items():
    with open(filename, 'r', encoding='utf-8') as f:
        content = f.read()
        
        # Split content by standalone "QUESTION" lines
        # Pattern: "QUESTION" on its own line (preceded by newline or start of string, followed by newline)
        question_blocks = re.split(r'(?:^|\n)QUESTION\n', content, flags=re.MULTILINE)
        # First element is empty or content before first QUESTION, so skip it
        question_blocks = [block.strip() for block in question_blocks[1:] if block.strip()]
        
        for idx, question_block in enumerate(question_blocks, start=1):
            # Extract context (everything before "The question the students are answering is:")
            # This includes teaching assistant instruction and lab description, but NOT the question text
            context_match = re.search(r'(.*?)(?=The question the students are answering is:)', question_block, re.DOTALL)
            context = context_match.group(1).strip() if context_match else ""
            
            # Find "The rubric is:" and extract everything after it
            # This captures the complete rubric including all sections
            rubric_match = re.search(r'The rubric is:\s*\n(.*?)(?=\n\nQUESTION|$)', question_block, re.DOTALL)
            if not rubric_match:
                # Fallback: just get everything after "The rubric is:"
                rubric_match = re.search(r'The rubric is:\s*\n(.*)$', question_block, re.DOTALL)
            
            if rubric_match:
                rubric = rubric_match.group(1).strip()
                # Remove any trailing whitespace/newlines
                rubric = re.sub(r'\n+$', '', rubric)
                rubrics_dict[(lab_num, idx)] = rubric
                rubric_context_dict[(lab_num, idx)] = context

# Add rubric column to the dataframe
initial_df['rubric'] = initial_df.apply(
    lambda row: rubrics_dict.get((row['lab_number'], row['question_number']), ''), 
    axis=1
)

# Validation output: Show summary of extracted rubrics
validation_df = pd.DataFrame([
    {
        'lab_number': lab,
        'question_number': q,
        'rubric_length': len(rubric),
        'has_meets_expectation': 'Meets Expectation' in rubric,
        'has_does_not_meet': 'Does not Meet Expectation' in rubric,
        'rubric_preview': rubric[:150] + '...' if len(rubric) > 150 else rubric
    }
    for (lab, q), rubric in sorted(rubrics_dict.items())
])

print(f"Extracted {len(rubrics_dict)} rubrics\n")
print(validation_df.to_string(index=False))
print(f"\nRows with rubric: {(initial_df['rubric'] != '').sum()} out of {len(initial_df)}")


Extracted 25 rubrics

 lab_number  question_number  rubric_length  has_meets_expectation  has_does_not_meet                                                                                                                                              rubric_preview
          1                1            400                   True               True   Meets Expectation: Students correctly identify that the HAL is the Hardware Abstraction Layer. Students correctly identify that the HAL is used to sim...
          1                2            467                   True               True   Meets Expectation: Students correctly identify that the __io_putchar function is a helper function that enables printing to the UART or console. Stude...
          1                3            322                   True               True   Meets Expectation: Students correctly identify that the debugger was required in order to examine the values stored in the registers. They state that ...
          

In [8]:
print(initial_df[(initial_df['lab_number'] == 1) & (initial_df['question_number'] == 1)]['rubric'].unique())

['Meets Expectation: Students correctly identify that the HAL is the Hardware Abstraction Layer. Students correctly identify that the HAL is used to simplify interaction with the hardware. Studens provide some additional, correct details regarding why it is useful to have a HAL. \n\nDoes not Meet Expectation: The student work either contains errors, omissions, or is not written with sufficient detail.']


In [9]:
# Helper function: Extract quality dimensions from rubric
def extract_qds_from_rubric(rubric_text, context_text=""):
    """
    Extract quality dimensions from a rubric text using LLM.
    Args:
        rubric_text: The grading rubric text
        context_text: Context about the lab and question (lab description, question text)
    Returns list of QDs as dictionaries with 'name' and 'definition' keys.
    """
    print("  Calling LLM to extract QDs from rubric...")
    
    context_section = ""
    if context_text:
        context_section = f"\nCONTEXT:\n{context_text}\n\n"
    
    prompt = f"""Analyze this grading rubric and extract 3-5 binary quality dimensions.

{context_section}
RUBRIC:
{rubric_text}

A quality dimension should be:
- BINARY (present or absent, not a scale)
- DISTINGUISHABLE (can be reliably detected by reading the text)
- VARIABLE (present in some correct answers, absent in others)
- SPECIFIC to the rubric criteria (not generic like "clarity" or "completeness")

Focus on CONCEPTUAL criteria from the rubric, not writing style.
{"Use the context above to understand the domain and terminology." if context_text else ""}

Output ONLY a valid JSON array with NO additional text:
[
  {{
    "name": "short_dimension_name",
    "definition": "clear definition of what presence means"
  }}
]"""
    
    response = generate_with_delay(prompt, reasoning=False, delay=1.5)
    
    # Parse JSON response
    try:
        # Try to extract JSON array from response
        import re
        json_match = re.search(r'\[.*\]', response, re.DOTALL)
        if json_match:
            qds = json.loads(json_match.group(0))
            if isinstance(qds, list) and len(qds) >= 2:
                print(f"  Successfully extracted {len(qds)} QDs")
                return qds[:5]  # Limit to 5 QDs
    except Exception as e:
        print(f"  Failed to parse QDs: {e}")
    
    # Fallback QDs based on common rubric patterns
    print("  Using fallback QDs")
    return [
        {"name": "correct_identification", "definition": "correctly identifies the main concept or term"},
        {"name": "explains_purpose", "definition": "explains why or how the concept is used"},
        {"name": "provides_details", "definition": "includes specific details or examples"}
    ]

print("Function defined: extract_qds_from_rubric")


Function defined: extract_qds_from_rubric


In [10]:
# Helper function: Grade a student answer using original rubric
def grade_with_rubric(student_answer, rubric_text, question_prompt):
    """
    Grade a student answer using the original rubric (general text).
    Returns 0 (does not meet) or 1 (meets expectation).
    """
    prompt = f"""You are grading a student answer using a rubric.

QUESTION: {question_prompt}

RUBRIC:
{rubric_text}

STUDENT ANSWER:
{student_answer}

Based STRICTLY on the rubric criteria, does this answer meet expectations?

You must output ONLY:
Grade: 0
OR
Grade: 1

Grade must be 0 (does not meet expectation) or 1 (meets expectation).
Be consistent in your interpretation of the rubric."""
    
    response = generate_with_delay(prompt, reasoning=False, delay=1.0)
    
    # Parse grade - look for explicit grade marking
    if "Grade: 1" in response or "Grade:1" in response:
        return 1
    elif "Grade: 0" in response or "Grade:0" in response:
        return 0
    # Fallback: check for keywords
    elif "meets expectation" in response.lower() and "does not meet" not in response.lower():
        return 1
    else:
        return 0

print("Function defined: grade_with_rubric")


Function defined: grade_with_rubric


In [11]:
# Helper function: Grade a student answer using QDs
def grade_with_qds(student_answer, qds, question_prompt):
    """
    Grade a student answer using quality dimensions (structured criteria).
    Returns 0 (does not meet) or 1 (meets expectation).
    """
    qd_descriptions = "\n".join([f"- {qd['name']}: {qd['definition']}" for qd in qds])
    
    prompt = f"""You are grading a student answer using structured quality dimensions (QDs).

QUESTION: {question_prompt}

QUALITY DIMENSIONS (Binary - Present or Absent):
{qd_descriptions}

STUDENT ANSWER:
{student_answer}

For each QD, determine if it is PRESENT (1) or ABSENT (0) in the answer.
If MOST QDs are present, the answer meets expectations (Grade: 1).
If MOST QDs are absent, the answer does not meet expectations (Grade: 0).

You must output ONLY:
Grade: 0
OR
Grade: 1

Be objective and consistent based on QD presence."""
    
    response = generate_with_delay(prompt, reasoning=False, delay=1.0)
    
    # Parse grade - look for explicit grade marking
    if "Grade: 1" in response or "Grade:1" in response:
        return 1
    elif "Grade: 0" in response or "Grade:0" in response:
        return 0
    # Fallback: check for keywords
    elif "meets expectation" in response.lower() and "does not meet" not in response.lower():
        return 1
    else:
        return 0

print("Function defined: grade_with_qds")


Function defined: grade_with_qds


In [12]:
# Helper function: Calculate flip rate for a single answer
def calculate_answer_flip_rate(grades):
    """
    Calculate flip rate for a single answer graded multiple times.
    Returns True if grades are inconsistent (flip occurred), False otherwise.
    """
    unique_grades = len(set(grades))
    return unique_grades > 1

print("Function defined: calculate_answer_flip_rate")


Function defined: calculate_answer_flip_rate


In [ ]:
# Helper function: Analyze inconsistency patterns
def analyze_inconsistency(sample_answers, qd_results, current_qds):
    """
    Analyze which answers are flipping and why.
    Returns diagnostic information for refinement.
    """
    print("  Analyzing inconsistency patterns...")
    
    flip_cases = []
    for idx, result in qd_results.items():
        if result['has_flip']:
            flip_cases.append({
                'answer_idx': idx,
                'answer_text': sample_answers[idx],
                'grades': result['grades'],
                'stats': result.get('stats', {})
            })
    
    if not flip_cases:
        print("  No flips detected")
        return None
    
    print(f"  Found {len(flip_cases)} answers with flips")
    return flip_cases

# Helper function: Refine QDs to reduce inconsistency
def refine_qds(current_qds, rubric_text, sample_answers, qd_results, baseline_results=None):
    """
    Ask LLM to refine QDs to reduce grading inconsistency.
    Uses MERGE/SPLIT/ADD/DROP operators based on analysis.
    
    Args:
        current_qds: Current quality dimensions
        rubric_text: Original rubric text
        sample_answers: List of full answer texts
        qd_results: Dict mapping answer index to {'grades': [...], 'has_flip': bool, 'stats': {...}}
        baseline_results: Dict mapping answer index to baseline grading results (for comparison)
    
    Returns tuple: (refined_qds, operations_applied)
    """
    print("  Calling LLM to suggest QD refinements...")
    
    # Analyze what's causing flips
    flip_info = analyze_inconsistency(sample_answers, qd_results, current_qds)
    
    if not flip_info:
        print("  No inconsistency to fix")
        return current_qds, []
    
    # Build detailed QD comparison (previous vs current)
    qd_text = "\n".join([f"{i+1}. {qd['name']}: {qd['definition']}" 
                         for i, qd in enumerate(current_qds)])
    
    # Build comprehensive inconsistency examples with FULL answers and detailed grading patterns
    inconsistent_examples = []
    for i, f in enumerate(flip_info[:5], 1):  # Include up to 5 examples
        answer_idx = f['answer_idx']
        answer_text = sample_answers[answer_idx]
        grades = f['grades']
        
        # Calculate statistics for this answer
        stats = qd_results[answer_idx].get('stats', {})
        count_0 = stats.get('count_0', grades.count(0))
        count_1 = stats.get('count_1', grades.count(1))
        majority = stats.get('majority', 1 if count_1 > count_0 else 0)
        
        # Get baseline comparison if available - CRITICAL for preserving baseline majority
        baseline_info = ""
        baseline_majority_warning = ""
        if baseline_results and answer_idx in baseline_results:
            base = baseline_results[answer_idx]
            base_grades = base.get('grades', [])
            base_stats = base.get('stats', {})
            base_majority = base_stats.get('majority', '?')
            base_count_0 = base_stats.get('count_0', base_grades.count(0))
            base_count_1 = base_stats.get('count_1', base_grades.count(1))
            baseline_info = f"\n   Baseline: grades={base_grades}, majority={base_majority} (0s:{base_count_0}/1s:{base_count_1})"
            
            # Check if QD majority flipped from baseline
            if majority != base_majority:
                baseline_majority_warning = f"\n   ⚠ VIOLATION: QD majority ({majority}) FLIPPED from baseline ({base_majority}) - REFINED QDs MUST CORRECT THIS!"
        
        example_text = f"""
Example {i}:
Answer Text (FULL):
{answer_text}

QD Grading Results:
- Grades across {len(grades)} trials: {grades}
- Count of 0s: {count_0}, Count of 1s: {count_1}
- Majority vote: {majority} ({max(count_0, count_1)}/{len(grades)} = {stats.get('majority_ratio', 0.0):.1%})
- INCONSISTENCY: This answer received BOTH 0s and 1s across multiple grading attempts{baseline_info}{baseline_majority_warning}
"""
        inconsistent_examples.append(example_text)
    
    inconsistency_analysis = "".join(inconsistent_examples)
    
    prompt = f"""You are refining Quality Dimensions (QDs) to reduce grading inconsistency.

ORIGINAL RUBRIC:
{rubric_text}

PREVIOUS QUALITY DIMENSIONS ({len(current_qds)} QDs) - REFINE THESE:
{qd_text}

PROBLEM: These QDs cause INCONSISTENT grading - the same answer gets different grades (0 or 1) across multiple grading attempts.

INCONSISTENT CASES WITH DETAILED GRADING PATTERNS:
{inconsistency_analysis}

Your task is to REFINE the previous QDs by applying operators (MERGE/SPLIT/ADD/DROP/KEEP) to make grading more consistent.

CRITICAL REQUIREMENTS:
1. REFERENCE the PREVIOUS QDs above - you must refine FROM them
2. For each refined QD, indicate which PREVIOUS QD(s) it came from (use "from_qd" field)
3. PRESERVE BASELINE MAJORITY VOTES - answers that had majority 0 in baseline must stay majority 0, majority 1 must stay 1
4. If QD-based grading flipped a majority vote from baseline, the refined QDs MUST correct this to match baseline
5. The refined QDs should reduce inconsistency (same answer gets same grade across trials) while preserving baseline grading intent

REFINEMENT OPERATORS:
1. MERGE: Combine two or more QDs that always appear together or are redundant
2. SPLIT: Break apart a QD that is too broad and causes ambiguity
3. ADD: Add a new QD to capture missing criteria (reference which gap it fills)
4. DROP: Remove a QD that doesn't help distinguish or causes inconsistency
5. KEEP: Keep a QD that's working well (may refine definition slightly)

For each refined QD:
- "name": Short descriptive name
- "definition": Clear, binary criterion (present/absent)
- "operator": Which operator (MERGE/SPLIT/ADD/DROP/KEEP)
- "from_qd": Name(s) of previous QD(s) this came from (use "new" if ADD)
- "explanation": Why this refinement will reduce inconsistency

Output ONLY a valid JSON array with NO additional text:
[
  {{
    "name": "dimension_name",
    "definition": "clear, objective definition",
    "operator": "MERGE|SPLIT|ADD|DROP|KEEP",
    "from_qd": "previous_qd_name or 'new'",
    "explanation": "why this refinement addresses the inconsistency patterns shown above"
  }}
]"""
    
    response = generate_with_delay(prompt, reasoning=False, delay=2.0)
    
    # Parse JSON response
    try:
        import re
        json_match = re.search(r'\[.*\]', response, re.DOTALL)
        if json_match:
            qds = json.loads(json_match.group(0))
            if isinstance(qds, list) and len(qds) >= 2:
                print(f"  Successfully refined to {len(qds)} QDs")
                
                # Extract operations
                operations = []
                for qd in qds[:5]:
                    op = qd.get('operator', 'UNKNOWN')
                    from_qd = qd.get('from_qd', 'unknown')
                    explanation = qd.get('explanation', 'No explanation provided')
                    operations.append({
                        'qd_name': qd['name'],
                        'operator': op,
                        'from_qd': from_qd,
                        'explanation': explanation
                    })
                    print(f"    - {op}: {qd['name']} (from: {from_qd})")
                
                return qds[:5], operations
    except Exception as e:
        print(f"  Failed to parse refined QDs: {e}")
        print("  Keeping current QDs")
        return current_qds, operations
    
    return current_qds, operations

print("Function defined: analyze_inconsistency, refine_qds")


Function defined: analyze_inconsistency, refine_qds


In [14]:
# Helper function: Calculate per-answer statistics
def calculate_answer_stats(grades):
    """Calculate statistics for a single answer's grades"""
    count_0 = grades.count(0)
    count_1 = grades.count(1)
    total = len(grades)
    majority = 1 if count_1 > count_0 else 0
    majority_ratio = max(count_0, count_1) / total if total > 0 else 0.0
    return {
        'count_0': count_0,
        'count_1': count_1,
        'total': total,
        'majority': majority,
        'majority_ratio': majority_ratio
    }

# Main experiment function: Process a single (lab, question) pair
def process_question(lab_num, question_num, df_subset, rubric_text, question_prompt, num_trials=10, max_refinements=3, context_text=""):
    """
    Run the full experiment for one question:
    1. Baseline grading with rubric
    2. Extract QDs and grade with QDs
    3. Refine QDs if needed (up to max_refinements times)
    
    Args:
        context_text: Context about the lab and question (lab description, question text)
    
    Returns dictionary with results.
    """
    print(f"\n{'='*80}")
    print(f"Processing Lab {lab_num}, Question {question_num}")
    print(f"Number of student answers: {len(df_subset)}")
    print(f"{'='*80}")
    
    # Get unique student answers
    student_answers = df_subset['question_text'].unique().tolist()
    print(f"Unique student texts: {len(student_answers)}")
    
    # IMPORTANT: bad_questions.csv contains answers that had flips in the original data
    # We need to prioritize answers that actually had grade flips in the data
    flip_answers = []
    consistent_answers = []
    answer_to_usernames = {}  # Map answer text to list of usernames who submitted it
    
    for answer_text in student_answers:
        answer_subset = df_subset[df_subset['question_text'] == answer_text]
        # Get usernames for this answer
        answer_to_usernames[answer_text] = answer_subset['username'].unique().tolist()
        # If this answer has multiple submissions with different grades, it's a flip answer
        if len(answer_subset) > 1 and answer_subset['grade'].nunique() > 1:
            flip_answers.append(answer_text)
        else:
            consistent_answers.append(answer_text)
    
    print(f"Answers with flips: {len(flip_answers)}, Consistent answers: {len(consistent_answers)}")
    
    # Sample 40-50 answers per question, prioritizing flip answers
    target_sample_size = 45  # Target: ~45 answers per question
    max_sample_size = 50
    min_sample_size = 40
    
    if len(flip_answers) >= target_sample_size:
        # We have enough flip answers, sample from them
        sample_answers = random.sample(flip_answers, min(target_sample_size, len(flip_answers)))
    elif len(flip_answers) >= min_sample_size:
        # We have enough flip answers to reach minimum
        sample_answers = flip_answers.copy()
        # Add consistent answers to reach target
        remaining = target_sample_size - len(sample_answers)
        if len(consistent_answers) > 0 and remaining > 0:
            additional = random.sample(consistent_answers, min(remaining, len(consistent_answers)))
            sample_answers.extend(additional)
    else:
        # Not enough flip answers, take all flip answers and fill with consistent
        sample_answers = flip_answers.copy()
        remaining = target_sample_size - len(sample_answers)
        if len(consistent_answers) > 0 and remaining > 0:
            additional = random.sample(consistent_answers, min(remaining, len(consistent_answers)))
            sample_answers.extend(additional)
    
    # Ensure we don't exceed max_sample_size
    if len(sample_answers) > max_sample_size:
        sample_answers = random.sample(sample_answers, max_sample_size)
    
    # Ensure we have at least min_sample_size if possible
    if len(sample_answers) < min_sample_size and len(student_answers) >= min_sample_size:
        # Add more if we have more available
        available_answers = [a for a in student_answers if a not in sample_answers]
        needed = min_sample_size - len(sample_answers)
        if len(available_answers) >= needed:
            sample_answers.extend(random.sample(available_answers, needed))
    
    flip_count_in_sample = len([a for a in sample_answers if a in flip_answers])
    consistent_count_in_sample = len(sample_answers) - flip_count_in_sample
    
    print(f"Using {len(sample_answers)} answers for this experiment ({flip_count_in_sample} with flips, {consistent_count_in_sample} consistent)")
    
    # Map sample answers to their usernames
    sample_usernames = [answer_to_usernames.get(answer, []) for answer in sample_answers]
    
    # STEP 1: Baseline grading with original rubric
    print(f"\nSTEP 1: Baseline grading with original rubric ({num_trials} trials per answer)")
    baseline_results = {}
    baseline_flips = 0
    
    for idx, answer in enumerate(sample_answers):
        grades = []
        for trial in range(num_trials):
            grade = grade_with_rubric(answer, rubric_text, question_prompt)
            grades.append(grade)
        
        has_flip = calculate_answer_flip_rate(grades)
        stats = calculate_answer_stats(grades)
        baseline_results[idx] = {
            'grades': grades, 
            'has_flip': has_flip,
            'stats': stats
        }
        if has_flip:
            baseline_flips += 1
        
        print(f"  Answer {idx+1}: grades={grades}, flip={has_flip}, stats=0s:{stats['count_0']}/1s:{stats['count_1']}, majority={stats['majority']} ({stats['majority_ratio']:.1%})")
    
    baseline_flip_rate = (baseline_flips / len(sample_answers)) * 100
    print(f"Baseline flip rate: {baseline_flip_rate:.1f}% ({baseline_flips}/{len(sample_answers)} answers)")
    
    # STEP 2: Extract QDs from rubric (skip if baseline already perfect)
    if baseline_flip_rate == 0.0:
        print(f"\nBaseline already perfect (0% flip rate) - skipping QD development")
        best_qds = []
        best_qd_flip_rate = 0.0
        best_majority_violations = 0
        best_iteration = -1
        best_qd_results = {}
        all_operations = []
        iteration_qds = []
        improvement = 0.0
    else:
        print(f"\nSTEP 2: Extracting quality dimensions from rubric")
        current_qds = extract_qds_from_rubric(rubric_text, context_text)
        print(f"Extracted {len(current_qds)} QDs:")
        for qd in current_qds:
            print(f"  - {qd['name']}: {qd['definition']}")
        
        # STEP 3: Grade with QDs and refine if needed
        best_qds = current_qds
        best_qd_flip_rate = baseline_flip_rate  # Compare against baseline, not 100%
        best_majority_violations = 0  # Track best majority violations count
        best_qd_results = None
        best_iteration = -1  # Track which iteration was best (-1 = initial QDs, 0+ = refinement iteration)
        all_operations = []  # Track all operations applied
        iteration_qds = []  # Track QDs at each iteration
        
        # Track previous iteration's results for incremental comparison
        previous_iteration_results = baseline_results  # Start with baseline (original rubric)
    
        for refinement_iter in range(max_refinements + 1):
            if refinement_iter == 0:
                print(f"\nSTEP 3.{refinement_iter}: Initial QD-based grading ({num_trials} trials per answer)")
            else:
                print(f"\nSTEP 3.{refinement_iter}: Refinement iteration {refinement_iter} ({num_trials} trials per answer)")
            
            qd_results = {}
            qd_flips = 0
            
            for idx, answer in enumerate(sample_answers):
                grades = []
                for trial in range(num_trials):
                    grade = grade_with_qds(answer, current_qds, question_prompt)
                    grades.append(grade)
                
                has_flip = calculate_answer_flip_rate(grades)
                stats = calculate_answer_stats(grades)
                qd_results[idx] = {
                    'grades': grades, 
                    'has_flip': has_flip,
                    'stats': stats
                }
                if has_flip:
                    qd_flips += 1
                
                # Compare with baseline
                baseline_stats = baseline_results[idx]['stats']
                majority_match = "✓" if stats['majority'] == baseline_stats['majority'] else "✗"
                print(f"  Answer {idx+1}: grades={grades}, flip={has_flip}, stats=0s:{stats['count_0']}/1s:{stats['count_1']}, majority={stats['majority']} ({stats['majority_ratio']:.1%}) {majority_match}")
            
            qd_flip_rate = (qd_flips / len(sample_answers)) * 100
            print(f"QD flip rate: {qd_flip_rate:.1f}% ({qd_flips}/{len(sample_answers)} answers)")
            
            # Validate majority vote consistency with nuanced logic:
            # - ALWAYS check against ORIGINAL baseline (preserve original grading intent)
            # - Track per-answer changes vs previous iteration (for improvement tracking)
            majority_violations = 0  # Count violations vs baseline
            improved_answers = 0  # Count answers that improved vs previous iteration
            worsened_answers = 0  # Count answers that worsened vs previous iteration
            
            comparison_for_tracking = previous_iteration_results if refinement_iter > 0 else baseline_results
            comparison_label = "previous iteration" if refinement_iter > 0 else "baseline"
            
            for idx in qd_results.keys():
                if idx in baseline_results:
                    baseline_majority = baseline_results[idx]['stats']['majority']
                    qd_majority = qd_results[idx]['stats']['majority']
                    
                    # Check against baseline (always)
                    if baseline_majority != qd_majority:
                        majority_violations += 1
                    
                    # Track improvement/worsening vs previous iteration (for logging)
                    if idx in comparison_for_tracking:
                        prev_majority = comparison_for_tracking[idx]['stats']['majority']
                        prev_flip = comparison_for_tracking[idx]['has_flip']
                        curr_flip = qd_results[idx]['has_flip']
                        
                        # Answer improved: fixed a flip, or improved consistency
                        if prev_flip and not curr_flip:
                            improved_answers += 1
                        elif not prev_flip and curr_flip:
                            worsened_answers += 1
                        # If both have flips, compare consistency ratio
                        elif prev_flip and curr_flip:
                            prev_consistency = comparison_for_tracking[idx]['stats']['majority_ratio']
                            curr_consistency = qd_results[idx]['stats']['majority_ratio']
                            if curr_consistency > prev_consistency:
                                improved_answers += 1
                            elif curr_consistency < prev_consistency:
                                worsened_answers += 1
            
            if majority_violations > 0:
                print(f"  ⚠ WARNING: {majority_violations} answer(s) have flipped majority vote from original baseline!")
            
            # Log improvement tracking
            if refinement_iter > 0:
                if improved_answers > 0:
                    print(f"  → {improved_answers} answer(s) improved vs {comparison_label}")
                if worsened_answers > 0:
                    print(f"  → {worsened_answers} answer(s) worsened vs {comparison_label} (but validated against baseline)")
            
            # Track QDs at this iteration - save comprehensive information
            iteration_data = {
                'iteration': refinement_iter,
                'qds': [{'name': qd['name'], 'definition': qd['definition']} for qd in current_qds.copy()],
                'flip_rate': qd_flip_rate,
                'majority_violations': majority_violations,
                'is_best': False,  # Will be set after evaluation
                'sample_answers': sample_answers.copy(),  # Full answer texts
                'qd_results': {
                    idx: {
                        'grades': result['grades'].copy(),
                        'has_flip': result['has_flip'],
                        'stats': result['stats'].copy()
                    }
                    for idx, result in qd_results.items()
                },
                'comparison_baseline': "original baseline",  # Always compare majority votes against original baseline
                'comparison_results': {
                    idx: {
                        'grades': comp_result.get('grades', []).copy() if isinstance(comp_result.get('grades', []), list) else comp_result.get('grades'),
                        'stats': comp_result.get('stats', {}).copy()
                    }
                    for idx, comp_result in baseline_results.items()
                    if idx in qd_results
                }
            }
            iteration_qds.append(iteration_data)
            
            # Initialize best_iteration after first QD grading (iteration 0)
            if refinement_iter == 0 and best_iteration == -1:
                best_iteration = 0
                best_qd_results = qd_results
                # Update best_majority_violations based on first iteration
                best_majority_violations = majority_violations
            
            # Hybrid comparison strategy:
            # - Majority votes: ALWAYS compare against ORIGINAL baseline (preserve grading intent)
            # - Flip rate IMPROVEMENTS: Compare incrementally against PREVIOUS iteration (track progress)
            # - When violations exist: Compare flip rate against BASELINE (ensure we don't worsen)
            
            # Get previous iteration's flip rate (for incremental comparison)
            if refinement_iter == 0:
                previous_flip_rate = baseline_flip_rate
            else:
                # Get flip rate from previous iteration
                prev_flips = sum(1 for idx, result in previous_iteration_results.items() if result.get('has_flip', False))
                previous_flip_rate = (prev_flips / len(sample_answers)) * 100 if sample_answers else baseline_flip_rate
            
            is_improvement = False
            should_early_stop = False
            
            # Best case: No majority violations (matches baseline) AND flip rate improved from previous iteration
            if majority_violations == 0:
                if qd_flip_rate < previous_flip_rate:
                    # Improved incrementally from previous iteration
                    is_improvement = True
                    print(f"✓ IMPROVEMENT: Flip rate reduced from {previous_flip_rate:.1f}% to {qd_flip_rate:.1f}% (majority votes match baseline)")
                    # Update best if this is better than current best
                    if qd_flip_rate < best_qd_flip_rate or best_majority_violations > 0:
                        best_qd_flip_rate = qd_flip_rate
                        best_majority_violations = 0
                        best_qds = current_qds
                        best_qd_results = qd_results
                        best_iteration = refinement_iter
                    
                    # If we've eliminated all flips, stop refining
                    if qd_flip_rate == 0:
                        print("Perfect consistency achieved! Stopping refinement.")
                        break
                else:
                    # No improvement in flip rate from previous iteration
                    print(f"No improvement: {qd_flip_rate:.1f}% >= {previous_flip_rate:.1f}% (majority votes preserved from baseline)")
            
            # If we have majority violations (worsened relative to baseline)
            elif majority_violations > 0:
                # Compare flip rate against PREVIOUS iteration when violations exist (incremental improvement tracking)
                if majority_violations > best_majority_violations:
                    # More violations than best - this is worse
                    if best_majority_violations == 0:
                        # We went from 0 violations to violations - this is bad
                        should_early_stop = True
                        print(f"  ⚠ EARLY STOPPING: Refinement introduced {majority_violations} majority vote violation(s) from baseline (best had 0)")
                    else:
                        # We're getting worse - early stop
                        should_early_stop = True
                        print(f"  ⚠ EARLY STOPPING: Refinement worsened to {majority_violations} violations from baseline (best had {best_majority_violations})")
                elif majority_violations == best_majority_violations:
                    # Same violation count - accept if flip rate improved OR if net answer improvement
                    net_improvement = improved_answers - worsened_answers
                    
                    if qd_flip_rate < best_qd_flip_rate:
                        # Flip rate improved vs best so far
                        is_improvement = True
                        print(f"  ⚠ PARTIAL IMPROVEMENT: Flip rate improved to {qd_flip_rate:.1f}% vs best ({best_qd_flip_rate:.1f}%) with {majority_violations} violation(s) vs baseline")
                        best_qd_flip_rate = qd_flip_rate
                        best_qds = current_qds
                        best_qd_results = qd_results
                        best_iteration = refinement_iter
                    elif net_improvement > 0:
                        # More answers improved than worsened (even if flip rate same or slightly worse)
                        is_improvement = True
                        print(f"  ⚠ PARTIAL IMPROVEMENT: Net {net_improvement} answer improvement ({improved_answers} improved, {worsened_answers} worsened vs previous), same violations ({majority_violations})")
                        best_qd_flip_rate = qd_flip_rate  # Update even if same/worse (track current state)
                        best_qds = current_qds
                        best_qd_results = qd_results
                        best_iteration = refinement_iter
                    else:
                        print(f"  ⚠ Rejecting: Same violations ({majority_violations}), flip rate {qd_flip_rate:.1f}% >= best {best_qd_flip_rate:.1f}%, net improvement: {net_improvement}")
                elif majority_violations < best_majority_violations:
                    # FEWER violations than best - this is better! Accept even if flip rate is higher
                    is_improvement = True
                    print(f"✓ IMPROVEMENT: Reduced violations from {best_majority_violations} to {majority_violations} (flip rate: {qd_flip_rate:.1f}%)")
                    best_qd_flip_rate = qd_flip_rate
                    best_majority_violations = majority_violations
                    best_qds = current_qds
                    best_qd_results = qd_results
                    best_iteration = refinement_iter
                else:
                    print(f"  ⚠ Rejecting: {majority_violations} violations (best: {best_majority_violations}), flip rate: {qd_flip_rate:.1f}%")
            
            # Early stopping: If current iteration has significantly more violations, revert to best
            if should_early_stop:
                print(f"\n{'='*80}")
                print(f"EARLY STOPPING: Current refinement ({refinement_iter}) is worse than best iteration ({best_iteration})")
                print(f"  Best iteration: {best_iteration} (flip rate: {best_qd_flip_rate:.1f}%, violations: {best_majority_violations})")
                print(f"  Current iteration: {refinement_iter} (flip rate: {qd_flip_rate:.1f}%, violations: {majority_violations})")
                print(f"  Reverting to best QDs from iteration {best_iteration}")
                print(f"{'='*80}\n")
                # Restore best QDs and results
                current_qds = best_qds
                break
            
            # Update is_best flag after evaluation
            iteration_qds[-1]['is_best'] = (refinement_iter == best_iteration)
            
            # Update previous iteration results for next refinement
            previous_iteration_results = qd_results.copy()
            
            # Refine QDs if we haven't reached max iterations and still have flips
            if refinement_iter < max_refinements and qd_flip_rate > 0:
                print(f"\nRefining QDs (attempt {refinement_iter + 1}/{max_refinements})...")
                # Pass previous iteration results as baseline for comparison
                # For first refinement (iter 0->1), use original baseline; otherwise use previous QD results
                comparison_baseline = baseline_results if refinement_iter == 0 else previous_iteration_results
                comparison_label = "original rubric baseline" if refinement_iter == 0 else "previous iteration"
                
                # Save what we're passing to refinement for this iteration
                refinement_input = {
                    'iteration': refinement_iter + 1,  # This is the refinement number
                    'input_qds': [{'name': qd['name'], 'definition': qd['definition']} for qd in current_qds],
                    'sample_answers': sample_answers.copy(),
                    'qd_results_before_refinement': {
                        idx: {
                            'grades': result['grades'].copy(),
                            'has_flip': result['has_flip'],
                            'stats': result['stats'].copy()
                        }
                        for idx, result in qd_results.items()
                    },
                    'comparison_baseline': "original baseline",  # Always compare majority votes against original baseline
                    'comparison_results': {
                        idx: {
                            'grades': comp_result.get('grades', []).copy() if isinstance(comp_result.get('grades', []), list) else comp_result.get('grades'),
                            'stats': comp_result.get('stats', {}).copy()
                        }
                        for idx, comp_result in baseline_results.items()
                        if idx in qd_results
                    }
                }
                
                # For refinement guidance, compare against previous iteration (incremental improvement)
                print(f"  Comparing against: {comparison_label} (for refinement guidance)")
                current_qds, operations = refine_qds(
                    current_qds, 
                    rubric_text, 
                    sample_answers, 
                    qd_results,
                    baseline_results=comparison_baseline  # Pass previous iteration for refinement context
                )
                
                # Save the refinement input and operations
                if operations:
                    refinement_input['operations'] = operations.copy()
                    refinement_input['output_qds'] = [{'name': qd['name'], 'definition': qd['definition']} for qd in current_qds]
                    all_operations.append(refinement_input)
                else:
                    # Still save the refinement attempt even if no operations
                    all_operations.append(refinement_input)
                print(f"Refined to {len(current_qds)} QDs:")
                for qd in current_qds:
                    print(f"  - {qd['name']}: {qd['definition']}")
            else:
                break
        
        # Calculate improvement
        improvement = baseline_flip_rate - best_qd_flip_rate
    
    print(f"\n{'='*80}")
    print(f"FINAL RESULTS for Lab {lab_num}, Question {question_num}:")
    print(f"  Baseline flip rate: {baseline_flip_rate:.1f}%")
    print(f"  Best QD flip rate: {best_qd_flip_rate:.1f}%")
    print(f"  Best iteration: {best_iteration} ({'initial QDs' if best_iteration == -1 else f'refinement {best_iteration}'})")
    print(f"  Best majority violations: {best_majority_violations}")
    print(f"  Improvement: {improvement:+.1f} percentage points")
    
    # ONE-TO-ONE MAPPING: Show per-answer comparison
    print(f"\n{'='*80}")
    print("ONE-TO-ONE ANSWER MAPPING:")
    print(f"{'='*80}")
    print(f"\n{'Answer':<8} {'Baseline':<25} {'QD Result':<25} {'Baseline Stats':<20} {'QD Stats':<20} {'Majority Match':<15}")
    print("-" * 120)
    
    for idx in sorted(baseline_results.keys()):
        base = baseline_results[idx]
        qd = best_qd_results.get(idx, {}) if best_qd_results else {}
        
        base_grades_str = str(base['grades'])
        qd_grades_str = str(qd.get('grades', [])) if qd else "N/A"
        
        base_stats = base.get('stats', {})
        base_stats_str = f"0s:{base_stats.get('count_0', 0)}/1s:{base_stats.get('count_1', 0)} (maj:{base_stats.get('majority', '?')})"
        
        qd_stats = qd.get('stats', {}) if qd else {}
        qd_stats_str = f"0s:{qd_stats.get('count_0', 0)}/1s:{qd_stats.get('count_1', 0)} (maj:{qd_stats.get('majority', '?')})" if qd_stats else "N/A"
        
        if qd_stats:
            majority_match = "✓ MATCH" if base_stats.get('majority') == qd_stats.get('majority') else "✗ FLIPPED"
        else:
            majority_match = "N/A"
        
        print(f"#{idx+1:<7} {base_grades_str:<25} {qd_grades_str:<25} {base_stats_str:<20} {qd_stats_str:<20} {majority_match:<15}")
    
    # Print operations summary
    if all_operations:
        print(f"\n{'='*80}")
        print(f"OPERATIONS APPLIED:")
        for iter_ops in all_operations:
            print(f"  Iteration {iter_ops['iteration']}:")
            for op in iter_ops['operations']:
                print(f"    - {op['operator']}: {op['qd_name']}")
                if op.get('explanation'):
                    print(f"      Reason: {op['explanation'][:100]}...")
    else:
        print(f"\n{'='*80}")
        print(f"OPERATIONS APPLIED: None (initial QDs were sufficient)")
    
    print(f"{'='*80}")
    
    # Log per-question metrics to MLflow (if in an active run)
    try:
        if mlflow.active_run():
            mlflow.log_metric(f"L{lab_num}Q{question_num}_baseline_flip_rate", baseline_flip_rate)
            mlflow.log_metric(f"L{lab_num}Q{question_num}_qd_flip_rate", best_qd_flip_rate)
            mlflow.log_metric(f"L{lab_num}Q{question_num}_improvement", improvement)
            mlflow.log_metric(f"L{lab_num}Q{question_num}_majority_violations", best_majority_violations)
            mlflow.log_metric(f"L{lab_num}Q{question_num}_best_iteration", best_iteration)
            mlflow.log_metric(f"L{lab_num}Q{question_num}_num_answers", len(sample_answers))
            mlflow.log_metric(f"L{lab_num}Q{question_num}_num_qds", len(best_qds))
            mlflow.log_metric(f"L{lab_num}Q{question_num}_num_refinements", len(all_operations))
            
            # Log QDs as text artifact
            qds_text = "\n\n".join([f"QD {i+1}: {qd['name']}\n{qd['definition']}" for i, qd in enumerate(best_qds)])
            import tempfile
            import os
            with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
                f.write(f"Lab {lab_num}, Question {question_num} - Final QDs\n")
                f.write("="*80 + "\n\n")
                f.write(qds_text)
                temp_path = f.name
            
            mlflow.log_artifact(temp_path, f"L{lab_num}Q{question_num}_qds.txt")
            os.unlink(temp_path)
    except Exception as e:
        # MLflow not available or not in active run - continue without logging
        pass
    
    return {
        'lab_number': lab_num,
        'question_number': question_num,
        'num_answers': len(sample_answers),
        'baseline_flip_rate': baseline_flip_rate,
        'qd_flip_rate': best_qd_flip_rate,
        'best_iteration': best_iteration,
        'best_majority_violations': best_majority_violations,
        'improvement': improvement,
        'qds_used': best_qds,  # Final best QDs
        'iteration_qds': iteration_qds,  # Comprehensive data at each iteration (QDs, answers, results, comparisons)
        'operations_applied': all_operations,  # Refinement inputs and operations for each refinement
        'baseline_results': baseline_results,
        'qd_results': best_qd_results,
        'sample_answers': sample_answers,  # Full answer texts
        'sample_usernames': sample_usernames,  # Usernames for each answer
        'rubric_text': rubric_text,  # Original rubric
        'context_text': context_text  # Lab/question context
    }

print("Function defined: process_question")


Function defined: process_question


In [15]:
# Enhanced logging and data saving function
def save_experiment_results(results, experiment_name="experiment"):
    """
    Save detailed experiment results to JSON files with comprehensive logging
    """
    import os
    from datetime import datetime
    
    # Create results directory
    results_dir = f"experiment_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    os.makedirs(results_dir, exist_ok=True)
    
    print(f"\nSaving results to: {results_dir}/")
    
    # Save summary
    summary = {
        'timestamp': datetime.now().isoformat(),
        'experiment_name': experiment_name,
        'total_questions': len(results),
        'avg_baseline_flip_rate': sum(r['baseline_flip_rate'] for r in results) / len(results) if results else 0,
        'avg_qd_flip_rate': sum(r['qd_flip_rate'] for r in results) / len(results) if results else 0,
        'avg_improvement': sum(r['improvement'] for r in results) / len(results) if results else 0,
        'improved_count': sum(1 for r in results if r['improvement'] > 0),
        'per_question_results': []
    }
    
    # Save per-question detailed data
    for r in results:
        # Get full answer texts and usernames
        sample_answers = r.get('sample_answers', [])
        sample_usernames = r.get('sample_usernames', [])
        
        # Per-answer results in the format requested
        per_answer = []
        for idx in sorted(r.get('baseline_results', {}).keys()):
            per_answer.append({
                'answer_index': idx,
                'question_text': sample_answers[idx] if idx < len(sample_answers) else '',  # Full answer text
                'usernames': sample_usernames[idx] if idx < len(sample_usernames) else [],  # Usernames who submitted this answer
                'baseline_scores': r['baseline_results'][idx]['grades'],
                'qd_scores': r['qd_results'][idx]['grades'] if 'qd_results' in r and r['qd_results'] else [],
                'baseline_flip': r['baseline_results'][idx]['has_flip'],
                'qd_flip': r['qd_results'][idx]['has_flip'] if 'qd_results' in r and r['qd_results'] else False,
                'borderline_flags': [False] * len(r['baseline_results'][idx]['grades'])
            })
        
        question_summary = {
            'lab': r['lab_number'],
            'question': r['question_number'],
            'baseline_flip_rate': r['baseline_flip_rate'],
            'qd_flip_rate': r['qd_flip_rate'],
            'improvement': r['improvement'],
            'num_answers': r['num_answers'],
            'operations': r.get('operations_applied', []),
            'final_qds': r.get('qds_used', []),  # Final best QDs
            'iteration_qds': r.get('iteration_qds', []),  # QDs at each iteration with flip rates
            'per_answer_results': per_answer
        }
        summary['per_question_results'].append(question_summary)
        
        # Save individual question result
        question_file = f"{results_dir}/L{r['lab_number']}Q{r['question_number']}_result.json"
        with open(question_file, 'w') as f:
            json.dump(question_summary, f, indent=2)
    
    # Save full summary
    summary_file = f"{results_dir}/summary.json"
    with open(summary_file, 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"✓ Saved {len(results)} question results")
    print(f"✓ Summary: {summary_file}")
    
    return results_dir

print("✓ Enhanced logging function defined")


✓ Enhanced logging function defined


In [16]:
# Run experiment on all questions with MLflow tracking
def run_full_experiment(df, rubrics_dict, num_trials=10, max_refinements=3, experiment_name="QD_Consistency_Experiment"):
    """
    Run the experiment on all (lab, question) pairs in the dataframe with MLflow tracking.
    """
    # Start MLflow run
    mlflow.set_experiment(experiment_name)
    
    with mlflow.start_run(run_name=f"experiment_{datetime.now().strftime('%Y%m%d_%H%M%S')}"):
        # Log all parameters
        mlflow.log_param("num_trials_per_answer", num_trials)
        mlflow.log_param("max_refinements", max_refinements)
        mlflow.log_param("api_url", BASE_URL)
        mlflow.log_param("timestamp", datetime.now().isoformat())
        
        print(f"\n{'#'*80}")
        print(f"# STARTING FULL EXPERIMENT (MLflow Run ID: {mlflow.active_run().info.run_id})")
        print(f"# Number of trials per answer: {num_trials}")
        print(f"# Max refinement iterations: {max_refinements}")
        print(f"{'#'*80}")
        
        all_results = []
        
        # Get unique (lab, question) pairs
        lab_question_pairs = df[['lab_number', 'question_number']].drop_duplicates().values.tolist()
        mlflow.log_param("total_questions", len(lab_question_pairs))
        print(f"\nTotal (lab, question) pairs to process: {len(lab_question_pairs)}")
        
        for lab_num, question_num in lab_question_pairs:
            # Get subset for this question
            df_subset = df[(df['lab_number'] == lab_num) & (df['question_number'] == question_num)]
            
            # Get rubric and context
            rubric_text = rubrics_dict.get((lab_num, question_num), '')
            context_text = rubric_context_dict.get((lab_num, question_num), '')
            if not rubric_text:
                print(f"\nSkipping Lab {lab_num}, Question {question_num}: No rubric found")
                continue
            
            # Get question prompt (use first row's context)
            question_prompt = f"Lab {lab_num}, Question {question_num}"
            
            # Process this question
            result = process_question(
                lab_num, 
                question_num, 
                df_subset, 
                rubric_text, 
                question_prompt,
                num_trials=num_trials,
                max_refinements=max_refinements,
                context_text=context_text
            )
            
            all_results.append(result)
    
        # Summary statistics
        print(f"\n{'#'*80}")
        print(f"# EXPERIMENT COMPLETE")
        print(f"{'#'*80}")
        print(f"\nProcessed {len(all_results)} questions")
        
        avg_baseline = sum(r['baseline_flip_rate'] for r in all_results) / len(all_results) if all_results else 0
        avg_qd = sum(r['qd_flip_rate'] for r in all_results) / len(all_results) if all_results else 0
        avg_improvement = sum(r['improvement'] for r in all_results) / len(all_results) if all_results else 0
        
        # Log summary metrics
        mlflow.log_metric("avg_baseline_flip_rate", avg_baseline)
        mlflow.log_metric("avg_qd_flip_rate", avg_qd)
        mlflow.log_metric("avg_improvement_pp", avg_improvement)
        
        # Count how many improved
        improved_count = sum(1 for r in all_results if r['improvement'] > 0)
        improvement_rate = (100*improved_count/len(all_results)) if all_results else 0
        mlflow.log_metric("questions_improved_count", improved_count)
        mlflow.log_metric("questions_improved_rate", improvement_rate)
        
        print(f"\nAVERAGE RESULTS:")
        print(f"  Average baseline flip rate: {avg_baseline:.1f}%")
        print(f"  Average QD flip rate: {avg_qd:.1f}%")
        print(f"  Average improvement: {avg_improvement:+.1f} percentage points")
        print(f"\nQuestions with improvement: {improved_count}/{len(all_results)} ({improvement_rate:.1f}%)")
        
        # Save and log summary JSON
        summary_data = {
            'experiment_summary': {
                'total_questions': len(all_results),
                'avg_baseline_flip_rate': avg_baseline,
                'avg_qd_flip_rate': avg_qd,
                'avg_improvement': avg_improvement,
                'improved_count': improved_count,
                'improvement_rate': improvement_rate
            },
            'per_question_results': [
                {
                    'lab': r['lab_number'],
                    'question': r['question_number'],
                    'baseline_flip_rate': r['baseline_flip_rate'],
                    'qd_flip_rate': r['qd_flip_rate'],
                    'improvement': r['improvement'],
                    'majority_violations': r.get('best_majority_violations', 0)
                }
                for r in all_results
            ]
        }
        
        # Log summary as artifact
        import tempfile
        import os
        with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
            json.dump(summary_data, f, indent=2)
            temp_path = f.name
        
        mlflow.log_artifact(temp_path, "summary.json")
        os.unlink(temp_path)
        
        # Log all results as artifact
        with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
            json.dump(all_results, f, indent=2, default=str)
            temp_path = f.name
        
        mlflow.log_artifact(temp_path, "all_results.json")
        os.unlink(temp_path)
        
        print(f"\n✓ MLflow Run ID: {mlflow.active_run().info.run_id}")
        print(f"✓ View results at: mlflow ui")
    
    return all_results

print("Function defined: run_full_experiment")


Function defined: run_full_experiment


## Approach Explanation

**Goal**: Demonstrate that using Quality Dimensions (QDs) leads to more consistent grading than using general rubrics.

**Method**:
1. **Baseline**: Grade each answer 10 times using the ORIGINAL RUBRIC (general text)
   - Calculate flip rate: % of answers that get inconsistent grades
2. **QD Extraction**: Extract binary quality dimensions from the rubric
3. **QD Grading**: Grade same answers 10 times using the EXTRACTED QDs (structured criteria)
   - Calculate flip rate with QD-based grading
4. **Refinement** (if needed): If QD flip rate is still high:
   - Analyze WHY inconsistency occurs
   - Apply MERGE/SPLIT/ADD/DROP operators to refine QDs
   - Re-grade and check if consistency improved
   - Repeat up to 3 times

**Key Insight**: QDs should be MORE CONSISTENT, not MORE STRICT. We're testing if structured quality dimensions reduce grading variance compared to general rubric text.


## Run Experiment

Now we can run the experiment. Start with a small test (e.g., 1-2 questions) to verify everything works before running on all questions.

Set parameters:
- `num_trials`: Number of times to grade each answer (default 10)
- `max_refinements`: Maximum number of QD refinement attempts (default 3)


In [17]:
# # TEST RUN: Run on first question only to verify setup
# # Uncomment to test:
# import random
# from datetime import datetime
# from time import sleep
# import time
# import json
# test_df = initial_df[(initial_df['lab_number'] == 1) & (initial_df['question_number'] == 1)]
# print(f"Test dataframe size: {len(test_df)}")

# result = process_question(
#     lab_num=1,
#     question_num=1,
#     df_subset=test_df,
#     rubric_text=rubrics_dict.get((1, 1), ''),
#     question_prompt="Lab 1, Question 1",
#     num_trials=3,  # Reduced for testing
#     max_refinements=2  # Reduced for testing
# )

# print("\nTest complete! Review output above before running full experiment.")


In [20]:
# Example usage with enhanced logging and MLflow tracking
# Run test on a few questions and save detailed results

test_questions = [
    (2,2),
]

# Set up MLflow experiment
mlflow.set_experiment("Manual_Test_Runs")

# Start MLflow run for this batch
with mlflow.start_run(run_name=f"manual_test_{datetime.now().strftime('%Y%m%d_%H%M%S')}"):
    # Log experiment parameters
    mlflow.log_param("num_questions", len(test_questions))
    mlflow.log_param("num_trials_per_answer", 10)
    mlflow.log_param("max_refinements", 2)
    mlflow.log_param("api_url", BASE_URL)
    mlflow.log_param("timestamp", datetime.now().isoformat())
    
    print(f"\n{'#'*80}")
    print(f"Testing {len(test_questions)} questions with enhanced logging")
    print(f"MLflow Run ID: {mlflow.active_run().info.run_id}")
    print(f"{'#'*80}\n")
    
    test_results = []
    for lab_num, question_num in test_questions:
        print(f"\nProcessing Lab {lab_num}, Question {question_num}...")
        
        # Get data
        df_subset = initial_df[(initial_df['lab_number'] == lab_num) & (initial_df['question_number'] == question_num)]
        rubric_text = rubrics_dict.get((lab_num, question_num), '')
        context_text = rubric_context_dict.get((lab_num, question_num), '')
        
        if not rubric_text:
            print(f"Skipping L{lab_num}Q{question_num}: No rubric")
            continue
        
        # Run experiment (MLflow tracking happens inside process_question)
        result = process_question(
            lab_num, 
            question_num, 
            df_subset, 
            rubric_text, 
            f"Lab {lab_num}, Question {question_num}",
            num_trials=10,  # Reduced for testing
            max_refinements=2,
            context_text=context_text
        )
        
        test_results.append(result)
    
    # Calculate summary metrics
    if test_results:
        avg_baseline = sum(r['baseline_flip_rate'] for r in test_results) / len(test_results)
        avg_qd = sum(r['qd_flip_rate'] for r in test_results) / len(test_results)
        avg_improvement = sum(r['improvement'] for r in test_results) / len(test_results)
        improved_count = sum(1 for r in test_results if r['improvement'] > 0)
        
        # Log summary metrics
        mlflow.log_metric("avg_baseline_flip_rate", avg_baseline)
        mlflow.log_metric("avg_qd_flip_rate", avg_qd)
        mlflow.log_metric("avg_improvement_pp", avg_improvement)
        mlflow.log_metric("questions_improved_count", improved_count)
        mlflow.log_metric("questions_improved_rate", 100*improved_count/len(test_results))
        
        # Log summary artifact
        summary_data = {
            'experiment_summary': {
                'total_questions': len(test_results),
                'avg_baseline_flip_rate': avg_baseline,
                'avg_qd_flip_rate': avg_qd,
                'avg_improvement': avg_improvement,
                'improved_count': improved_count
            },
            'per_question_results': [
                {
                    'lab': r['lab_number'],
                    'question': r['question_number'],
                    'baseline_flip_rate': r['baseline_flip_rate'],
                    'qd_flip_rate': r['qd_flip_rate'],
                    'improvement': r['improvement']
                }
                for r in test_results
            ]
        }
        
        import tempfile
        import os
        with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
            json.dump(summary_data, f, indent=2)
            temp_path = f.name
        
        mlflow.log_artifact(temp_path, "summary.json")
        os.unlink(temp_path)
        
        print(f"\nSummary:")
        print(f"  Average baseline flip rate: {avg_baseline:.1f}%")
        print(f"  Average QD flip rate: {avg_qd:.1f}%")
        print(f"  Average improvement: {avg_improvement:+.1f}pp")
        print(f"  Questions improved: {improved_count}/{len(test_results)}")
    
    # Save results with comprehensive logging
    results_dir = save_experiment_results(test_results, "test_experiment")
    print(f"\n{'#'*80}")
    print(f"✓ Results saved to: {results_dir}")
    print(f"✓ MLflow Run ID: {mlflow.active_run().info.run_id}")
    print(f"✓ View at: mlflow ui")
    print(f"{'#'*80}")



################################################################################
Testing 1 questions with enhanced logging
MLflow Run ID: 9743582d19bb4dad82f1e6a76cfa12b7
################################################################################


Processing Lab 2, Question 2...

Processing Lab 2, Question 2
Number of student answers: 100
Unique student texts: 23
Answers with flips: 23, Consistent answers: 0
Using 23 answers for this experiment (23 with flips, 0 consistent)

STEP 1: Baseline grading with original rubric (10 trials per answer)
  Answer 1: grades=[1, 1, 1, 1, 1, 1, 1, 1, 1, 1], flip=False, stats=0s:0/1s:10, majority=1 (100.0%)
  Answer 2: grades=[1, 1, 1, 1, 1, 1, 1, 1, 1, 1], flip=False, stats=0s:0/1s:10, majority=1 (100.0%)
  Answer 3: grades=[1, 1, 1, 1, 1, 1, 1, 1, 1, 1], flip=False, stats=0s:0/1s:10, majority=1 (100.0%)
  Answer 4: grades=[1, 1, 1, 1, 1, 1, 1, 1, 1, 1], flip=False, stats=0s:0/1s:10, majority=1 (100.0%)
  Answer 5: grades=[1, 1, 1, 1, 1, 1, 1,

KeyError: 'operations'

In [ ]:
# Comprehensive visualization of results
def visualize_experiment_results(result):
    """
    Create a publication-ready visualization of the experiment results.
    Shows: flip rate comparison, per-answer grades, operations applied, and QDs used.
    """
    print("\n" + "="*100)
    print(f"EXPERIMENT RESULTS: Lab {result['lab_number']}, Question {result['question_number']}")
    print("="*100)
    
    # Section 1: Overall Metrics
    print("\n" + "-"*100)
    print("1. OVERALL FLIP RATE COMPARISON")
    print("-"*100)
    
    print(f"\n{'Metric':<40} {'Baseline':<20} {'QD-Based':<20} {'Change':<20}")
    print("-"*100)
    print(f"{'Flip Rate':<40} {result['baseline_flip_rate']:>6.1f}% {result['qd_flip_rate']:>18.1f}% {result['improvement']:>14.1f}pp")
    print(f"{'Consistency Rate':<40} {100-result['baseline_flip_rate']:>6.1f}% {100-result['qd_flip_rate']:>18.1f}% {-(result['improvement']):>14.1f}pp")
    print(f"{'Number of Answers':<40} {result['num_answers']:>6} {result['num_answers']:>18} {'--':<20}")
    
    # Section 2: Per-Answer Grading
    print("\n" + "-"*100)
    print("2. PER-ANSWER GRADING RESULTS")
    print("-"*100)
    
    print(f"\n{'Answer':<10} {'Baseline Grades':<25} {'QD Grades':<25} {'Baseline Status':<20} {'QD Status':<20} {'Change':<15}")
    print("-"*100)
    
    for idx in sorted(result['baseline_results'].keys()):
        base_grades = result['baseline_results'][idx]['grades']
        qd_grades = result['qd_results'][idx]['grades']
        base_flip = result['baseline_results'][idx]['has_flip']
        qd_flip = result['qd_results'][idx]['has_flip']
        
        base_str = str(base_grades)
        qd_str = str(qd_grades)
        
        base_status = "FLIP" if base_flip else "Consistent"
        qd_status = "FLIP" if qd_flip else "Consistent"
        
        if base_flip and not qd_flip:
            change = "FIXED"
        elif not base_flip and not qd_flip:
            # Check if grades changed
            if base_grades == qd_grades:
                change = "Same"
            else:
                change = "Grade changed"
        elif base_flip and qd_flip:
            change = "Still flips"
        else:
            change = "Worsened"
        
        print(f"#{idx+1:<9} {base_str:<25} {qd_str:<25} {base_status:<20} {qd_status:<20} {change:<15}")
    
    # Section 3: Operations Applied
    print("\n" + "-"*100)
    print("3. REFINEMENT OPERATIONS APPLIED")
    print("-"*100)
    
    if 'operations_applied' in result and result['operations_applied']:
        for iter_ops in result['operations_applied']:
            print(f"\nRefinement Iteration {iter_ops['iteration']}:")
            print("-"*100)
            
            # Count operators
            op_counts = {}
            for op in iter_ops['operations']:
                op_type = op['operator']
                op_counts[op_type] = op_counts.get(op_type, 0) + 1
            
            print(f"\nOperator Summary: {', '.join([f'{k}: {v}' for k, v in sorted(op_counts.items())])}")
            print()
            
            for i, op in enumerate(iter_ops['operations'], 1):
                print(f"{i}. QD: {op['qd_name']}")
                print(f"   Operator: {op['operator']}")
                print(f"   Rationale: {op['explanation']}")
                print()
    else:
        print("\nNo operations applied - initial QDs were sufficient.")
    
    # Section 4: Final Quality Dimensions
    print("-"*100)
    print("4. FINAL QUALITY DIMENSIONS USED")
    print("-"*100)
    
    print(f"\nTotal QDs: {len(result['qds_used'])}\n")
    
    for i, qd in enumerate(result['qds_used'], 1):
        operator = qd.get('operator', 'N/A')
        print(f"{i}. {qd['name']} [{operator}]")
        print(f"   Definition: {qd['definition']}")
        if 'explanation' in qd:
            print(f"   Why {operator}: {qd['explanation'][:150]}...")
        print()
    
    # Section 5: Summary Statistics
    print("-"*100)
    print("5. SUMMARY STATISTICS")
    print("-"*100)
    
    # Calculate grade changes
    grade_changes = 0
    flip_fixes = 0
    for idx in result['baseline_results'].keys():
        if result['baseline_results'][idx]['has_flip'] and not result['qd_results'][idx]['has_flip']:
            flip_fixes += 1
        if result['baseline_results'][idx]['grades'] != result['qd_results'][idx]['grades']:
            grade_changes += 1
    
    print(f"\nFlips fixed: {flip_fixes}/{result['num_answers']}")
    print(f"Answers with grade changes: {grade_changes}/{result['num_answers']}")
    print(f"Improvement: {result['improvement']:+.1f} percentage points")
    
    if result['qd_flip_rate'] == 0:
        print("\nRESULT: Perfect consistency achieved!")
    elif result['qd_flip_rate'] < result['baseline_flip_rate']:
        print(f"\nRESULT: Flip rate reduced by {result['improvement']:.1f}pp")
    else:
        print("\nRESULT: No improvement in flip rate")
    
    print("\n" + "="*100 + "\n")

print("Function defined: visualize_experiment_results")


Function defined: visualize_experiment_results


In [ ]:
# Visualize the result with your data
visualize_experiment_results(result)



EXPERIMENT RESULTS: Lab 1, Question 5

----------------------------------------------------------------------------------------------------
1. OVERALL FLIP RATE COMPARISON
----------------------------------------------------------------------------------------------------

Metric                                   Baseline             QD-Based             Change              
----------------------------------------------------------------------------------------------------
Flip Rate                                   4.8%                0.0%            4.8pp
Consistency Rate                           95.2%              100.0%           -4.8pp
Number of Answers                            21                 21 --                  

----------------------------------------------------------------------------------------------------
2. PER-ANSWER GRADING RESULTS
----------------------------------------------------------------------------------------------------

Answer     Baseline Grades

In [ ]:
# Test on multiple specific questions
def test_multiple_questions(lab_question_pairs, num_trials=3, max_refinements=2):
    """
    Test the experiment on specific (lab, question) pairs.
    
    Args:
        lab_question_pairs: List of tuples [(lab_num, question_num), ...]
        num_trials: Number of grading trials per answer
        max_refinements: Maximum refinement iterations
    
    Returns:
        List of results dictionaries
    """
    results = []
    
    print(f"\n{'#'*100}")
    print(f"# TESTING {len(lab_question_pairs)} QUESTIONS")
    print(f"# Trials per answer: {num_trials}")
    print(f"# Max refinements: {max_refinements}")
    print(f"{'#'*100}\n")
    
    for lab_num, question_num in lab_question_pairs:
        # Get subset for this question
        df_subset = initial_df[(initial_df['lab_number'] == lab_num) & 
                                (initial_df['question_number'] == question_num)]
        
        # Get rubric
        rubric_text = rubrics_dict.get((lab_num, question_num), '')
        if not rubric_text:
            print(f"Skipping Lab {lab_num}, Question {question_num}: No rubric found\n")
            continue
        
        # Get question prompt
        question_prompt = f"Lab {lab_num}, Question {question_num}"
        
        # Process this question
        result = process_question(
            lab_num, 
            question_num, 
            df_subset, 
            rubric_text, 
            question_prompt,
            num_trials=num_trials,
            max_refinements=max_refinements
        )
        
        results.append(result)
    
    # Summary
    print(f"\n{'#'*100}")
    print(f"# TEST COMPLETE: {len(results)} questions processed")
    print(f"{'#'*100}\n")
    
    # Quick summary table
    print(f"{'Lab':<6} {'Q#':<6} {'Baseline %':<12} {'QD %':<12} {'Improvement':<15} {'Operations':<20}")
    print("-"*100)
    
    for r in results:
        ops_count = len(r['operations_applied'][0]['operations']) if r.get('operations_applied') else 0
        ops_summary = f"{ops_count} ops" if ops_count > 0 else "None"
        
        print(f"{r['lab_number']:<6} {r['question_number']:<6} {r['baseline_flip_rate']:>10.1f}% "
              f"{r['qd_flip_rate']:>10.1f}% {r['improvement']:>12.1f}pp  {ops_summary:<20}")
    
    # Overall stats
    if results:
        avg_baseline = sum(r['baseline_flip_rate'] for r in results) / len(results)
        avg_qd = sum(r['qd_flip_rate'] for r in results) / len(results)
        avg_improvement = sum(r['improvement'] for r in results) / len(results)
        
        print("-"*100)
        print(f"{'AVERAGE':<6} {'':>6} {avg_baseline:>10.1f}% {avg_qd:>10.1f}% {avg_improvement:>12.1f}pp")
        print("\n")
    
    return results

print("Function defined: test_multiple_questions")


Function defined: test_multiple_questions


## Understanding Operations

When refinement is needed, the system will apply one or more of these operators:

| Operator | Purpose | When Applied | Example |
|----------|---------|--------------|---------|
| **KEEP** | No change | QD is working well | Keep as-is |
| **MERGE** | Combine coupled QDs | Two QDs always appear together | Merge "mentions_energy" + "mentions_charge" → "energy_per_charge" |
| **SPLIT** | Break apart broad QD | QD covers multiple concepts | Split "explains_pointers" → "address_concept" + "operators" + "use_cases" |
| **ADD** | Introduce new QD | Missing aspect causing flips | Add "provides_example" if examples help consistency |
| **DROP** | Remove QD | QD doesn't discriminate (appears in all/no answers) | Drop "mentions_voltage" if 98% of answers have it |

### In Your Test Run:

- **Baseline**: 20% flip rate (1 out of 5 answers had inconsistent grades)
- **QD Extraction**: Successfully extracted 4 QDs from rubric
- **QD Grading**: 0% flip rate (all 5 answers graded consistently)
- **Operations**: **None needed** - immediate success!

This means the initial QDs were already well-defined and objective enough to eliminate grading inconsistency.


## Sample Size Calculation for Statistical Significance

This section helps determine how many questions and trials you need to achieve statistical significance.


## Analysis Functions

Functions to analyze and visualize the results after the experiment completes.


## Running Experiments with MLflow

All experiments are now automatically tracked in MLflow. Simply call `run_full_experiment()` and everything will be logged.


## Summary of Implementation

### What This Code Does

This notebook implements an experiment to test if **Quality Dimensions (QDs)** lead to more consistent grading than general rubric text.

### Key Functions

1. **`extract_qds_from_rubric(rubric_text)`** - Extracts binary quality dimensions from rubric
2. **`grade_with_rubric(answer, rubric, prompt)`** - Grades using original rubric text
3. **`grade_with_qds(answer, qds, prompt)`** - Grades using structured QDs  
4. **`refine_qds(qds, rubric, answers, results)`** - Uses MERGE/SPLIT/ADD/DROP to refine QDs
5. **`process_question(...)`** - Main experiment loop for one question
6. **`run_full_experiment(...)`** - Runs experiment on all questions
7. **`visualize_results(result)`** - Shows comparison between rubric vs QD grading

### Alignment with Research

- **Baseline**: General rubric text (current approach)
- **Treatment**: Structured QDs (proposed approach)
- **Metric**: Flip rate reduction (consistency improvement)
- **Refinement**: MERGE/SPLIT/ADD/DROP operators from Section 4 of Operations.md
- **Goal**: Show QDs are MORE CONSISTENT, not more strict

### Important Points

- QDs should make grading **consistent**, not **harder**
- Flip rate = % of answers with grade disagreements across trials
- Lower flip rate = more consistent grading = better
- Refinement focuses on making QDs more objective and distinguishable


In [ ]:
# ============================================================================
# COMPREHENSIVE ANALYSIS: Load JSON files and generate all tables/analysis
# ============================================================================

import json
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict
import re
from scipy.stats import pearsonr, spearmanr

# Set style
try:
    plt.style.use('seaborn-v0_8')
except:
    plt.style.use('seaborn')
sns.set_palette("husl")

# Results directory
RESULTS_DIR = Path('./experiment_results_20251103_034838')

print("=" * 80)
print("LOADING EXPERIMENT RESULTS FROM JSON FILES")
print("=" * 80)

# Load summary
with open(RESULTS_DIR / 'summary.json', 'r') as f:
    summary = json.load(f)

# Load individual question results
question_files = {
    'L1Q1': 'L1Q1_result.json',
    'L1Q2': 'L1Q2_result.json',
    'L1Q3': 'L1Q3_result.json',
    'L1Q4': 'L1Q4_result.json',
    'L1Q5': 'L1Q5_result.json',
}

question_data = {}
for q_name, filename in question_files.items():
    filepath = RESULTS_DIR / filename
    if filepath.exists():
        with open(filepath, 'r') as f:
            question_data[q_name] = json.load(f)
        print(f"✓ Loaded {q_name}")

print(f"\nLoaded {len(question_data)} question results")
print(f"Summary: {summary['total_questions']} questions")
print(f"Average baseline flip rate: {summary['avg_baseline_flip_rate']:.2f}%")
print(f"Average QD flip rate: {summary['avg_qd_flip_rate']:.2f}%")
print(f"Average improvement: {summary['avg_improvement']:.2f}%")
print("=" * 80)


LOADING EXPERIMENT RESULTS FROM JSON FILES
✓ Loaded L1Q1
✓ Loaded L1Q2
✓ Loaded L1Q3
✓ Loaded L1Q4
✓ Loaded L1Q5

Loaded 5 question results
Summary: 4 questions
Average baseline flip rate: 18.25%
Average QD flip rate: 11.21%
Average improvement: 7.04%


In [ ]:
# ============================================================================
# EXTRACT ALL METRICS FROM JSON FILES
# ============================================================================

def extract_question_metrics(q_data, q_name):
    """Extract all relevant metrics from a question's result data"""
    metrics = {
        'question_id': q_name,
        'lab': q_data.get('lab'),
        'question': q_data.get('question'),
        'num_answers': q_data.get('num_answers', 0),
        'baseline_flip_rate': q_data.get('baseline_flip_rate', 0),
        'qd_flip_rate': q_data.get('qd_flip_rate', 0),
        'improvement': q_data.get('improvement', 0),
        'relative_improvement': 0,
        'num_operations': len(q_data.get('operations', [])),
        'num_qds_initial': 0,
        'num_qds_final': 0,
        'avg_answer_length': 0,
        'answer_length_std': 0,
        'baseline_consistency': 0,
        'num_baseline_flips': 0,
        'num_qd_flips': 0,
        'num_fixed_by_qd': 0,
    }
    
    # Extract QD information
    operations = q_data.get('operations', [])
    if operations:
        first_op = operations[0]
        metrics['num_qds_initial'] = len(first_op.get('input_qds', []))
        
        # Get final QDs
        if 'final_qds' in q_data and q_data['final_qds']:
            metrics['num_qds_final'] = len(q_data['final_qds'])
        elif operations:
            last_op = operations[-1]
            if 'refined_qds' in last_op:
                metrics['num_qds_final'] = len(last_op['refined_qds'])
            else:
                metrics['num_qds_final'] = metrics['num_qds_initial']
    
    # Calculate answer characteristics
    per_answer = q_data.get('per_answer_results', [])
    if per_answer:
        answer_lengths = []
        baseline_consistencies = []
        baseline_flips = 0
        qd_flips = 0
        fixed_by_qd = 0
        
        for answer in per_answer:
            # Answer length
            question_text = answer.get('question_text', '')
            answer_lengths.append(len(question_text))
            
            # Baseline consistency
            baseline_scores = answer.get('baseline_scores', [])
            baseline_flip = answer.get('baseline_flip', False)
            qd_flip = answer.get('qd_flip', False)
            
            if baseline_scores:
                majority = max(set(baseline_scores), key=baseline_scores.count)
                consistency = baseline_scores.count(majority) / len(baseline_scores)
                baseline_consistencies.append(consistency)
            
            if baseline_flip:
                baseline_flips += 1
                if not qd_flip:
                    fixed_by_qd += 1
            
            if qd_flip:
                qd_flips += 1
        
        if answer_lengths:
            metrics['avg_answer_length'] = np.mean(answer_lengths)
            metrics['answer_length_std'] = np.std(answer_lengths)
        
        if baseline_consistencies:
            metrics['baseline_consistency'] = np.mean(baseline_consistencies)
        
        metrics['num_baseline_flips'] = baseline_flips
        metrics['num_qd_flips'] = qd_flips
        metrics['num_fixed_by_qd'] = fixed_by_qd
    
    # Calculate relative improvement
    if metrics['baseline_flip_rate'] > 0:
        metrics['relative_improvement'] = (metrics['improvement'] / metrics['baseline_flip_rate']) * 100
    
    return metrics

# Extract metrics for all questions
print("\n" + "=" * 80)
print("EXTRACTING METRICS FROM ALL QUESTIONS")
print("=" * 80)

all_metrics = []
for q_name, q_data in question_data.items():
    metrics = extract_question_metrics(q_data, q_name)
    all_metrics.append(metrics)

# Create DataFrame
metrics_df = pd.DataFrame(all_metrics)
metrics_df = metrics_df.sort_values('question_id')

# Display main metrics table
print("\nTABLE 1: Question Performance Metrics")
print("=" * 80)
display_cols = ['question_id', 'num_answers', 'baseline_flip_rate', 'qd_flip_rate', 
                'improvement', 'relative_improvement', 'num_qds_initial']
print(metrics_df[display_cols].to_string(index=False))
print("\n")



EXTRACTING METRICS FROM ALL QUESTIONS

TABLE 1: Question Performance Metrics
question_id  num_answers  baseline_flip_rate  qd_flip_rate  improvement  relative_improvement  num_qds_initial
       L1Q1            5            0.000000      0.000000     0.000000              0.000000                0
       L1Q2           14           35.714286     35.714286     0.000000              0.000000                3
       L1Q3           18           11.111111      5.555556     5.555556             50.000000                3
       L1Q4           28           21.428571      3.571429    17.857143             83.333333                3
       L1Q5           21            4.761905      0.000000     4.761905            100.000000                4




In [ ]:
# ============================================================================
# COMPREHENSIVE SUMMARY TABLES AND STATISTICAL ANALYSIS
# ============================================================================

print("\n" + "=" * 80)
print("COMPREHENSIVE ANALYSIS TABLES")
print("=" * 80)

# TABLE 3: Detailed Flip Rate Analysis
print("\nTABLE 3: Detailed Flip Rate Analysis")
print("=" * 80)
flip_cols = ['question_id', 'num_answers', 'num_baseline_flips', 'num_qd_flips', 
             'num_fixed_by_qd', 'baseline_flip_rate', 'qd_flip_rate', 'improvement']
flip_df = metrics_df[flip_cols].copy()
flip_df['fix_rate'] = (flip_df['num_fixed_by_qd'] / flip_df['num_baseline_flips'] * 100).round(1)
flip_df.loc[flip_df['num_baseline_flips'] == 0, 'fix_rate'] = 0
print(flip_df.to_string(index=False))

# TABLE 4: Overall Summary Statistics
print("\n\nTABLE 4: Overall Summary Statistics")
print("=" * 80)
overall_stats = {
    'Metric': [
        'Average Baseline Flip Rate (%)',
        'Average QD Flip Rate (%)',
        'Average Improvement (pp)',
        'Relative Improvement (%)',
        'Total Questions',
        'Questions Improved',
        'Questions with No Improvement',
        'Total Answers Analyzed'
    ],
    'Value': [
        f"{summary['avg_baseline_flip_rate']:.2f}",
        f"{summary['avg_qd_flip_rate']:.2f}",
        f"{summary['avg_improvement']:.2f}",
        f"{(summary['avg_improvement'] / summary['avg_baseline_flip_rate'] * 100):.1f}",
        f"{summary['total_questions']}",
        f"{summary['improved_count']}",
        f"{summary['total_questions'] - summary['improved_count']}",
        f"{metrics_df['num_answers'].sum()}"
    ]
}
summary_df = pd.DataFrame(overall_stats)
print(summary_df.to_string(index=False))

# TABLE 5: Question-by-Question Breakdown
print("\n\nTABLE 5: Question-by-Question Breakdown")
print("=" * 80)
breakdown_cols = ['question_id', 'num_answers', 'baseline_flip_rate', 'qd_flip_rate', 
                  'improvement', 'relative_improvement', 'num_qds_initial', 
                  'num_baseline_flips', 'num_qd_flips', 'num_fixed_by_qd']
breakdown_df = metrics_df[breakdown_cols].copy()
breakdown_df['improvement_status'] = breakdown_df['improvement'].apply(
    lambda x: 'Improved' if x > 0 else 'No Change' if x == 0 else 'Worsened'
)
print(breakdown_df.to_string(index=False))

print("\n" + "=" * 80)



COMPREHENSIVE ANALYSIS TABLES

TABLE 3: Detailed Flip Rate Analysis
question_id  num_answers  num_baseline_flips  num_qd_flips  num_fixed_by_qd  baseline_flip_rate  qd_flip_rate  improvement  fix_rate
       L1Q1            5                   0             0                0            0.000000      0.000000     0.000000       0.0
       L1Q2           14                   5             5                1           35.714286     35.714286     0.000000      20.0
       L1Q3           18                   2             1                1           11.111111      5.555556     5.555556      50.0
       L1Q4           28                   6             1                5           21.428571      3.571429    17.857143      83.3
       L1Q5           21                   1             0                1            4.761905      0.000000     4.761905     100.0


TABLE 4: Overall Summary Statistics
                        Metric Value
Average Baseline Flip Rate (%) 18.25
      Average QD Flip